# 07 Timing-informed share model (exploratory extension)

Re-runs the timing model on 300 municipalities, 2010–2023 (**2024 stays locked**).

| Model | What it adds to B4s |
|---|---|
| S1 | same-year curve of the *other* municipalities of the state (target left out) |
| S2 | same-year Google Trends "dengue" for the state |
| S3 | S1 + S2 |
| S3C | S3 + the 20 M1 climate features |
| T1 | B4s shape shifted by the state curve's timing |
| checks (design A) | wrong-year (`-wy`), non-study pool (`S1-ns`), 25 % / 50 % pools (`-p25`, `-p50`) |

References fitted in the same folds: B1, B4s, M1-noONI, M1.

**How to use (Colab):**
1. Runtime → *Run all*. It asks once for Google Drive access.
2. **Everything is saved on Google Drive** (`MyDrive/dengue_recon_data` and `MyDrive/dengue_recon_runs/timing`). If Colab disconnects, open the notebook again and *Run all*: finished folds are skipped.
3. Keep the browser tab open. A GPU is not needed.

**Time [uncertain]:** the first data download takes a few hours (only once; it resumes). Each fold has about 20 models, so design A (12 folds) and design C (14 folds) will likely take many hours on free Colab, over several sessions.


In [ ]:
# ---- Setup (Colab: clones the repo and saves everything on Google Drive) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "timing-model"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    os.environ.setdefault("DATA_DIR", str(DRIVE / "dengue_recon_data"))
    os.environ.setdefault("TIMING_WORK_DIR", str(DRIVE / "dengue_recon_runs" / "timing"))
else:
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("TIMING_TRENDS_DIR", str(REPO / "data_snapshots" / "google_trends"))  # committed snapshot
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from dengue_recon import config as C
from dengue_recon.extensions.timing import run as TR, signals as SG
C.ensure_dirs()
assert C.RUN_FINAL_2024 is False, "2024 must stay locked"
print("DATA_DIR        :", C.DATA_DIR)
print("checkpoints/out :", TR.work_dir())
print("Trends files    :", SG.trends_dir())


## 1. Data (once; saved on Drive)

Downloads InfoDengue (about 700 municipalities: the 300 study ones plus the others used for the state curves), NASA POWER weather and ONI, then builds the processed tables. It can be interrupted and resumed.

Note: InfoDengue revises past counts, so today's download can differ slightly from the October one. B4s and M1 are re-fitted in the same folds, so the comparisons inside this run stay fair; cell 4 also compares B4s with the official number.

In [ ]:
need = ["dengue_weekly.parquet", "climate_weekly.parquet", "municipalities.csv", "oni_weekly.parquet"]
n_info = len(list((C.RAW_DIR / "infodengue").glob("*.csv")))
missing = [f for f in need if not (C.DATA_DIR / f).exists()]
print("missing processed files:", missing, "| InfoDengue CSVs:", n_info)
if missing or n_info < 600:
    subprocess.run([sys.executable, str(C.ROOT / "scripts" / "run_pipeline.py"), "--step", "data"], check=True)
print("data ready:", all((C.DATA_DIR / f).exists() for f in need),
      "| InfoDengue CSVs:", len(list((C.RAW_DIR / "infodengue").glob("*.csv"))))


## 2. Google Trends

The repository has a snapshot downloaded on 3 Nov 2026 (`data_snapshots/google_trends/`). Use it, so results are reproducible: a new download gives slightly different numbers (Google samples its data).

If the files are missing, set `USE_TRENDS = False`: the run then drops S2, S3, S3C and their checks.

In [ ]:
USE_TRENDS = True
REDOWNLOAD_TRENDS = False   # True only if the snapshot is missing; slow (rate limits)

if REDOWNLOAD_TRENDS:
    from dengue_recon.extensions.timing import trends_download
    trends_download.main(["--out", str(DRIVE / "dengue_recon_data" / "google_trends" if IN_COLAB else SG.trends_dir())])
    if IN_COLAB:
        os.environ["TIMING_TRENDS_DIR"] = str(DRIVE / "dengue_recon_data" / "google_trends")
n_tr = len(list(SG.trends_dir().glob("dengue_*.csv")))
print("Trends files:", n_tr)
if USE_TRENDS and n_tr == 0:
    USE_TRENDS = False
    print("No Trends files -> running without the Trends models")


## 3. Run designs A and C

Design A first (primary), then C. Each finished fold is saved on Drive at once; *Run all* again after a disconnect and it continues.

In [ ]:
import time
for design in ["A", "C"]:
    t0 = time.time()
    TR.main(design=design, subset=300, climate="power", with_trends=USE_TRENDS)
    print(f"design {design} finished in {(time.time() - t0) / 3600:.1f} h")


## 4. Results summary

Mean JSD, peak-week error and onset-week error per model (lower is better), the gain over B4s, and the split by number of training years (design A: k ≤ 6 vs k ≥ 7, as for INLA).

Official references (design A, 300): B4s 0.1190, M1-noONI 0.1175, INLA-scaled 0.1126, INLA-scaled-clim15 0.1110.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.width", 200)
sfx = "" if USE_TRENDS else "_notrends"
for design in ["A", "C"]:
    tag = f"timing_design{design}_power_300{sfx}"
    f = TR.work_dir() / tag / f"metrics_detail_{tag}.csv"
    if not f.exists():
        print("not finished yet:", f); continue
    df = pd.read_csv(f)
    display(Markdown(f"### Design {design}"))
    mm = df.groupby("model", sort=False)[["jsd", "peak_err", "onset_err", "emd"]].mean()
    mm["jsd_vs_B4s_%"] = 100 * (mm.jsd / mm.loc["B4s", "jsd"] - 1)
    display(mm.round(4))
    if design == "A":
        df["k_group"] = (df.k >= 7).map({False: "k<=6 (2012-16)", True: "k>=7 (2017-23)"})
        display(df.pivot_table(index="model", columns="k_group", values="jsd", aggfunc="mean", sort=False).round(4))
    yr = df.pivot_table(index="year", columns="model", values="jsd", aggfunc="mean")
    better = {m: f"{int((yr[m] < yr['B4s']).sum())}/{len(yr)}" for m in yr if m != "B4s"}
    display(Markdown("Years better than B4s: " + ", ".join(f"{m} {v}" for m, v in better.items())))
    chk = TR.work_dir() / tag / f"checks_{tag}.json"
    if chk.exists():
        import json; c = json.loads(chk.read_text())
        print("leakage audit pass:", c["official_leakage_audit_all_pass"], "| feature audit pass:", c["feature_audit_all_pass"])


## 5. Send the results back

The full outputs are on Drive in `MyDrive/dengue_recon_runs/timing/`. The cell below makes one small zip (no large `.npz` files) in the same folder. Share that zip or the folder for the write-up and the commit.

In [ ]:
import shutil, tempfile
src = TR.work_dir()
tmp = Path(tempfile.mkdtemp()) / "timing_results"
shutil.copytree(src, tmp, ignore=shutil.ignore_patterns("*.npz"))
z = shutil.make_archive(str(src / "timing_results_small"), "zip", tmp)
print("zip:", z, f"({Path(z).stat().st_size / 1e6:.1f} MB)")
